# 01 — Domain Rental Data Curation

## Purpose
Clean and standardise the course-provided Domain.com.au Victorian rental listing dataset.

Input:
- `data/raw/vic_rentals_all.csv`

Output:
- `data/curated/vic_rentals.parquet`

This notebook:
- validates the raw schema
- checks duplicates and missingness
- standardises data types
- cleans categorical and feature fields
- flags suspicious observations
- produces a reproducible curated dataset

## Imports 


In [62]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

## Define Paths

In [63]:
RAW_PATH = Path("../data/raw/vic_rentals_all.csv")
CURATED_PATH = Path("../data/curated/vic_rentals.parquet")

assert RAW_PATH.exists(), f"Raw data not found: {RAW_PATH}"

CURATED_PATH.parent.mkdir(parents=True, exist_ok=True)

## Load raw data 

In [64]:
raw = pd.read_csv(RAW_PATH)

print(f"Rows: {raw.shape[0]:,}")
print(f"Columns: {raw.shape[1]}")

Rows: 12,717
Columns: 30


## Inpsect 

In [65]:
df = raw.copy()

pd.DataFrame({
    "dtype": df.dtypes,
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique": df.nunique(dropna=True)
})

,dtype,missing,missing_pct,unique
listing_id,int64,0,0.00,12717
suburb,object,0,0.00,656
postcode,int64,0,0.00,426
weekly_rent,float64,279,2.19,408
bond,float64,782,6.15,983
available_date,object,136,1.07,734
date_listed,object,4,0.03,655
days_listed,float64,4,0.03,661
bedrooms,float64,125,0.98,11
bathrooms,float64,51,0.40,9


### Initial observations

- `listing_id` is complete and unique and can be used as the listing-level identifier.
- Rental and core property variables have relatively low missingness.
- `carspaces` and `structured_features` have moderate missingness.
- `property_id` has substantial missingness and is unsuitable as the primary identifier.
- `land_area` is almost entirely missing and is unlikely to be useful downstream.
- Date columns require conversion from strings to datetime.
- `postcode` should be treated as a categorical/location identifier rather than a numeric measurement.
- Latitude and longitude are almost complete, making spatial assignment to SA2 feasible.

## Validate required columns

In [66]:
required_columns = {
    "listing_id",
    "suburb",
    "postcode",
    "weekly_rent",
    "bond",
    "available_date",
    "date_listed",
    "days_listed",
    "bedrooms",
    "bathrooms",
    "carspaces",
    "property_type",
    "address",
    "lat",
    "lon",
    "scraped_date",
    "structured_features"
}

missing_columns = required_columns - set(df.columns)

assert not missing_columns, f"Missing required columns: {missing_columns}"

## Check duplicates

In [67]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate listing IDs:", df["listing_id"].duplicated().sum())
print("Unique listing IDs:", df["listing_id"].nunique())


Exact duplicate rows: 0
Duplicate listing IDs: 0
Unique listing IDs: 12717


## Standardise text fields

In [68]:
text_columns = [
    "suburb",
    "address",
    "property_type",
    "primary_type",
    "secondary_type",
    "agency",
    "agent_names"
]

for col in text_columns:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

## Standardise postcode

In [69]:
df["postcode"] = (
    pd.to_numeric(df["postcode"], errors="coerce")
    .astype("Int64")
    .astype("string")
    .str.zfill(4)
)

## Clean numeric columns

In [70]:
numeric_columns = [
    "weekly_rent",
    "bond",
    "days_listed",
    "bedrooms",
    "bathrooms",
    "carspaces",
    "photo_count",
    "video_count",
    "floorplans_count",
    "lat",
    "lon"
]

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

## Convert integer-like fields

In [71]:
integer_columns = [
    "bedrooms",
    "bathrooms",
    "carspaces",
    "photo_count",
    "video_count",
    "floorplans_count",
    "days_listed"
]

for col in integer_columns:
    if col in df.columns:
        df[col] = df[col].round().astype("Int64")

## Parse dates

In [72]:
df["date_listed"] = pd.to_datetime(
    df["date_listed"],
    errors="coerce"
)

df["scraped_date"] = pd.to_datetime(
    df["scraped_date"],
    errors="coerce"
)

df["available_date"] = pd.to_datetime(
    df["available_date"],
    errors="coerce"
)

df[
    ["date_listed", "available_date", "scraped_date"]
].describe()

,date_listed,available_date,scraped_date
count,12713,12581,12717
mean,2025-06-21 22:43:18.977424640,2025-07-10 08:37:41.696208640,2025-09-09 16:41:58.622866944
min,2008-01-15 00:00:00,1970-01-01 00:00:00,2025-09-09 16:13:04
25%,2025-07-30 00:00:00,2025-08-13 00:00:00,2025-09-09 16:26:02
50%,2025-08-21 00:00:00,2025-09-03 00:00:00,2025-09-09 16:39:08
75%,2025-09-02 00:00:00,2025-09-19 00:00:00,2025-09-09 16:53:58
max,2025-09-09 00:00:00,2026-10-03 00:00:00,2025-09-09 18:07:20


## Recalculate days listed 

In [73]:
df["days_listed_calc"] = (
    df["scraped_date"].dt.normalize()
    - df["date_listed"].dt.normalize()
).dt.days

comparison = (
    df["days_listed"].astype("Float64")
    - df["days_listed_calc"].astype("Float64")
)

comparison.value_counts(dropna=False).head()

 0.0     10813
-1.0      1900
 <NA>        4
Name: count, dtype: Int64

### Days-listed validation

The supplied `days_listed` field was compared with the calendar-day difference between `scraped_date` and `date_listed`.

- 10,813 observations matched exactly.
- 1,900 observations differed by exactly one day.
- 4 observations could not be compared due to missing listing dates.

The one-day discrepancies are consistent with the loss of time-of-day information in `date_listed`. The supplied `days_listed` field is therefore retained.

## Flag long/impossible dates

In [74]:
df["flag_future_listed"] = (
    df["date_listed"] > df["scraped_date"]
)

df["flag_long_listing"] = df["days_listed"] > 365

df["flag_invalid_available_date"] = (
    df["available_date"].dt.year < 2020
)

## Inspect rental prices

In [75]:
df["weekly_rent"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
)

count     12438.000000
mean        764.713378
std        9428.719738
min           0.000000
1%          221.850000
5%          360.000000
25%         485.000000
50%         560.000000
75%         690.000000
95%        1100.000000
99%        1750.000000
max      808500.000000
Name: weekly_rent, dtype: float64

The weekly rent distribution is strongly right-skewed. While the median weekly rent is $560 and 99% of listings are below $1,750, the maximum observed value is $808,500. These extreme values substantially inflate the mean and standard deviation and are unlikely to represent genuine weekly residential rents. Suspicious rent observations should therefore be flagged and inspected before downstream analysis, while robust statistics such as the median are preferred for summarising rental prices.


In [76]:
df.nlargest(
    30,
    "weekly_rent"
)[
    [
        "listing_id",
        "weekly_rent",
        "property_type",
        "secondary_type",
        "bedrooms",
        "suburb",
        "address",
        "url"
    ]
]

,listing_id,weekly_rent,property_type,secondary_type,bedrooms,suburb,address,url
2586,17697966,808500.0,House,House,4,BERWICK,83 Golf Links Road,https://www.domain.com.au/83-golf-links-road-b...
3641,17750082,595000.0,House,House,3,ROSEBUD,4 Wilfred Street,https://www.domain.com.au/4-wilfred-street-ros...
1929,17747509,315000.0,Apartment / Unit / Flat,Apartment / Unit / Flat,1,MORWELL,1/28 Elgin Street,https://www.domain.com.au/1-28-elgin-street-mo...
10356,17712191,10000.0,House,House,6,TOORAK,<NA>,https://www.domain.com.au/toorak-vic-3142-1771...
12294,17661725,5866.0,House,House,5,CAMBERWELL,54 Fairmont Avenue,https://www.domain.com.au/54-fairmont-avenue-c...
10358,17722803,5750.0,House,House,4,TOORAK,2 Lisbuoy Court,https://www.domain.com.au/2-lisbuoy-court-toor...
12705,9287841,5000.0,Apartment / Unit / Flat,Apartment / Unit / Flat,5,BRIGHTON,<NA>,https://www.domain.com.au/brighton-vic-3186-92...
8776,17744678,4500.0,House,House,5,HAWTHORN,94 Illawarra Road,https://www.domain.com.au/94-illawarra-road-ha...
10768,17694334,4500.0,Apartment / Unit / Flat,Apartment / Unit / Flat,3,SOUTH YARRA,1 Fairlie Court,https://www.domain.com.au/1-fairlie-court-sout...
1236,17460975,4000.0,Apartment / Unit / Flat,Apartment / Unit / Flat,4,SOUTH MELBOURNE,901/161 Eastern Road,https://www.domain.com.au/901-161-eastern-road...


Inspection of the upper tail shows that most high weekly rents are associated with plausible premium properties in suburbs such as Toorak, Brighton, South Yarra, Docklands, and Port Melbourne. Values in the range of approximately $3,000–$10,000 per week may therefore represent genuine luxury rental listings and should not be removed solely because they are large.

However, three observations — $315,000, $595,000, and $808,500 per week — are orders of magnitude larger than the rest of the distribution and are inconsistent with plausible weekly residential rents. These values likely reflect data-entry or scraping errors, such as sale prices being captured as rental prices. They should be flagged for exclusion or correction in downstream rent-based analysis rather than applying a blanket upper threshold to all expensive listings.


In [77]:
df.nsmallest(
    30,
    "weekly_rent"
)[
    [
        "listing_id",
        "weekly_rent",
        "property_type",
        "secondary_type",
        "bedrooms",
        "suburb",
        "address",
        "url"
    ]
]

,listing_id,weekly_rent,property_type,secondary_type,bedrooms,suburb,address,url
3111,17159699,0.0,House,House,3,CLYDE,24 Viola Circuit,https://www.domain.com.au/24-viola-circuit-cly...
4353,10344261,0.0,House,House,2,NORTHCOTE,40 Charles Street,https://www.domain.com.au/40-charles-street-no...
5087,16697431,0.0,House,House,2,QUARRY HILL,11 Pyke Street,https://www.domain.com.au/11-pyke-street-quarr...
5091,17175722,0.0,House,House,2,QUARRY HILL,158 Mitchell Street,https://www.domain.com.au/158-mitchell-street-...
5092,17442602,0.0,House,House,4,QUARRY HILL,30 Echuca Street,https://www.domain.com.au/30-echuca-street-qua...
5887,16706257,0.0,House,House,4,NORTH BENDIGO,16 Michelsen Street,https://www.domain.com.au/16-michelsen-street-...
5892,17416575,0.0,Townhouse,Townhouse,3,NORTH BENDIGO,1/28 Bayne Street,https://www.domain.com.au/1-28-bayne-street-no...
7012,15367349,0.0,House,House,4,MERNDA,21 William Street,https://www.domain.com.au/21-william-street-me...
7014,16248581,0.0,House,House,3,MERNDA,8 Muir Way,https://www.domain.com.au/8-muir-way-mernda-vi...
8305,17316329,0.0,House,House,2,BENDIGO,221 Barnard Street,https://www.domain.com.au/221-barnard-street-b...


Inspection of the lower tail shows that very low weekly rents have mixed interpretations. Several observations with rents between approximately $35 and $50 correspond to car spaces or addresses explicitly referring to parking, so these values may be legitimate despite appearing unusually low.

In contrast, values such as $0–$40 attached to houses, townhouses, or apartments are implausible as normal residential weekly rents and are likely to represent missing, incorrectly parsed, or otherwise invalid rental prices. This indicates that low-rent validation should consider both the rental value and the listing type rather than applying a single numerical cutoff across all listings.

## Inspect non-standard property listings


In [78]:
df["secondary_type"].value_counts(dropna=False)

secondary_type
House                            6587
Apartment / Unit / Flat          4516
Townhouse                        1276
Studio                            209
New Apartments / Off the Plan      45
Villa                              28
Acreage / Semi-Rural               13
Semi-Detached                      10
New House & Land                    8
Car Space                           7
<NA>                                4
Duplex                              3
Block of Units                      3
Terrace                             3
Farm                                2
retirement                          2
Vacant land                         1
Name: count, dtype: Int64

In [79]:
df["primary_type"].value_counts(dropna=False)

primary_type
House               6598
Apartment           4728
Townhouse/Villa     1304
New Developments      45
Any                   37
<NA>                   4
Land                   1
Name: count, dtype: Int64

In [80]:
standard_residential_types = {
    "House",
    "Apartment / Unit / Flat",
    "Townhouse",
    "Studio",
    "Villa",
    "Semi-Detached",
    "Duplex",
    "Block of Units",
    "Terrace"
}

df["is_standard_residential"] = (
    df["secondary_type"].isin(standard_residential_types)
)

In [81]:
df["is_standard_residential"].value_counts()

is_standard_residential
True     12635
False       82
Name: count, dtype: int64

## Flag suspicious rents

In [82]:
df["flag_rent_suspicious"] = (
    df["is_standard_residential"]
    & (
        (df["weekly_rent"] < 100)
        | (df["weekly_rent"] > 10_000)
    )
)

## Inpsect bedrooms, bathrooms, and carspaces

In [83]:
for col in ["bedrooms", "bathrooms", "carspaces"]:
    print(f"\n{col}")
    print(
        df[col]
        .value_counts(dropna=False)
        .sort_index()
    )


bedrooms
bedrooms
1       1834
2       3416
3       4054
4       2973
5        261
6         28
7         10
8          8
9          6
11         1
50         1
<NA>     125
Name: count, dtype: Int64

bathrooms
bathrooms
1       5991
2       6026
3        575
4         55
5          9
6          3
7          1
9          5
12         1
<NA>      51
Name: count, dtype: Int64

carspaces
carspaces
1       5099
2       5023
3        369
4        294
5         52
6         36
7          6
8         10
9          4
10         9
11         2
12         3
14         2
15         1
19         1
20         2
22         1
<NA>    1803
Name: count, dtype: Int64


In [84]:
df.loc[
    (df["bedrooms"] >= 10)
    | (df["bathrooms"] >= 8)
    | (df["carspaces"] >= 10),
    [
        "listing_id",
        "weekly_rent",
        "bedrooms",
        "bathrooms",
        "carspaces",
        "property_type",
        "secondary_type",
        "suburb",
        "address",
        "url"
    ]
].sort_values(
    ["bedrooms", "bathrooms", "carspaces"],
    ascending=False
)

,listing_id,weekly_rent,bedrooms,bathrooms,carspaces,property_type,secondary_type,suburb,address,url
5362,15800260,1050.0,50,1,1,Apartment / Unit / Flat,Apartment / Unit / Flat,SOUTHBANK,1 Freshwater Place,https://www.domain.com.au/1-freshwater-place-s...
2133,17748261,1800.0,11,6,5,House,House,WOODSTOCK,1300 Donnybrook Road,https://www.domain.com.au/1300-donnybrook-road...
6949,17027842,3300.0,9,9,8,House,House,NEWTOWN,<NA>,https://www.domain.com.au/newtown-vic-3220-170...
2011,17737763,300.0,9,9,3,House,House,WAURN PONDS,27 Danaher Avenue,https://www.domain.com.au/27-danaher-avenue-wa...
7423,17735877,375.0,9,9,<NA>,House,House,WERRIBEE,78 Market Rd,https://www.domain.com.au/78-market-rd-werribe...
10347,17288084,NaN,6,5,10,House,House,TOORAK,<NA>,https://www.domain.com.au/toorak-vic-3142-1728...
11259,17735862,2000.0,5,4,15,House,House,WARRANDYTE,287-301 Jumping Creek Rd,https://www.domain.com.au/287-301-jumping-cree...
10464,17739856,895.0,5,2,12,House,House,NAR NAR GOON,5 Latta Road,https://www.domain.com.au/5-latta-road-nar-nar...
5599,16991827,650.0,5,2,10,House,House,CRAIGIEBURN,6 Plumpton Avenue,https://www.domain.com.au/6-plumpton-avenue-cr...
10491,17510840,1150.0,4,2,11,House,House,RYE,82 CREEDMORE DRIVE,https://www.domain.com.au/82-creedmore-drive-r...


## Flag suspicious property attributes

In [85]:
df["flag_bedrooms_suspicious"] = (
    df["bedrooms"] > 20
)

df["flag_bathrooms_suspicious"] = (
    (df["bathrooms"] > 10)
    | (
        (df["bathrooms"] >= 8)
        & (df["bedrooms"] <= 2)
    )
)

df["flag_carspaces_suspicious"] = (
    (df["carspaces"] >= 10)
    & df["secondary_type"].isin([
        "Apartment / Unit / Flat",
        "Studio"
    ])
)

df["flag_property_features_suspicious"] = (
    df["flag_bedrooms_suspicious"]
    | df["flag_bathrooms_suspicious"]
    | df["flag_carspaces_suspicious"]
)

property_flags = [
    "flag_bedrooms_suspicious",
    "flag_bathrooms_suspicious",
    "flag_carspaces_suspicious",
    "flag_property_features_suspicious"
]

for col in property_flags:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))


flag_bedrooms_suspicious
flag_bedrooms_suspicious
False    12591
<NA>       125
True         1
Name: count, dtype: Int64

flag_bathrooms_suspicious
flag_bathrooms_suspicious
False    12663
<NA>        51
True         3
Name: count, dtype: Int64

flag_carspaces_suspicious
flag_carspaces_suspicious
False    11526
<NA>      1182
True         9
Name: count, dtype: Int64

flag_property_features_suspicious
flag_property_features_suspicious
False    11456
<NA>      1248
True        13
Name: count, dtype: Int64


In [86]:
df.loc[
    df["flag_property_features_suspicious"],
    [
        "listing_id", "bedrooms", "bathrooms", "carspaces",
        "secondary_type", "suburb", "address"
    ] + property_flags
]

,listing_id,bedrooms,bathrooms,carspaces,secondary_type,suburb,address,flag_bedrooms_suspicious,flag_bathrooms_suspicious,flag_carspaces_suspicious,flag_property_features_suspicious
1279,17030693,1,9,<NA>,House,BELL POST HILL,51 Braund Avenue,False,True,False,True
4299,8431375,1,1,20,Apartment / Unit / Flat,BURWOOD,390 Burwood Highway,False,False,True,True
4300,8431378,1,1,12,Apartment / Unit / Flat,BURWOOD,224 Burwood Highway,False,False,True,True
4301,8431379,1,1,11,Apartment / Unit / Flat,BURWOOD,308 Burwood Highway,False,False,True,True
5362,15800260,50,1,1,Apartment / Unit / Flat,SOUTHBANK,1 Freshwater Place,True,False,False,True
6353,10621148,1,1,19,Studio,ST ALBANS,1A & 1B/101-103 Main Road West,False,False,True,True
6821,8431398,<NA>,1,10,Apartment / Unit / Flat,HAWTHORN EAST,2 Eastern Place,<NA>,False,True,True
6909,17753508,1,12,8,Studio,GEELONG WEST,6 Aberdeen St,False,True,False,True
6948,16916036,1,9,8,House,NEWTOWN,258 Latrobe Terrace,False,True,False,True
8827,8431400,<NA>,1,14,Apartment / Unit / Flat,HAWTHORN,1 Queens Avenue,<NA>,False,True,True


The 50-bedroom apartment and one-bedroom listings with very high bathroom or parking counts warrant review. Large houses can reasonably have many bedrooms, bathrooms or car spaces, so contextual flags are preferable to blanket cutoffs. All rows are retained; flags remain unknown where missing attributes prevent a decision.

## Coordinate validation

In [87]:
df[["lat", "lon"]].describe()

,lat,lon
count,12713.000000,12713.000000
mean,-37.781359,144.971809
std,0.388812,0.569393
min,-38.829080,141.000550
25%,-37.904865,144.870390
50%,-37.823673,144.982800
75%,-37.750935,145.111970
max,-34.166810,149.756800


In [88]:
df.loc[
    df[["lat", "lon"]].isna().any(axis=1),
    ["listing_id", "address", "suburb", "lat", "lon"]
]

,listing_id,address,suburb,lat,lon
10179,17727626,<NA>,Rowville,NaN,NaN
11405,17721987,<NA>,Irymple,NaN,NaN
11605,17722092,<NA>,Coimadai,NaN,NaN
11606,17727546,<NA>,Coimadai,NaN,NaN


In [89]:
df["flag_bad_coordinates"] = (
    df["lat"].isna()
    | df["lon"].isna()
    | ~df["lat"].between(-40, -33)
    | ~df["lon"].between(140, 150)
)

print(df["flag_bad_coordinates"].value_counts(dropna=False))
df.loc[
    df["flag_bad_coordinates"],
    ["listing_id", "address", "suburb", "lat", "lon"]
]

flag_bad_coordinates
False    12713
True         4
Name: count, dtype: int64


,listing_id,address,suburb,lat,lon
10179,17727626,<NA>,Rowville,NaN,NaN
11405,17721987,<NA>,Irymple,NaN,NaN
11605,17722092,<NA>,Coimadai,NaN,NaN
11606,17727546,<NA>,Coimadai,NaN,NaN


Coordinates are nearly complete. This broad bounding-box check identifies missing or out-of-range values; it does not establish that each point lies within Victoria. Coordinates will support SA2 spatial assignment in the dedicated geospatial notebook.

## Land area

In [90]:
print(df["land_area"].describe())
print(f"Missing land area: {df['land_area'].isna().mean():.2%}")
df.loc[df["land_area"].notna(), ["listing_id", "land_area"]]

count          2
unique         2
top       189 m²
freq           1
Name: land_area, dtype: object
Missing land area: 99.98%


,listing_id,land_area
4075,17725252,189 m²
12109,17733960,530 m2


Only two listings have land area recorded, leaving approximately 99.98% missing. Coverage is insufficient for meaningful downstream analysis, so this variable is removed.

In [91]:
df = df.drop(columns=["land_area"])

## Missing-value policy

In [92]:
df["features_missing"] = df["structured_features"].isna()

df["coordinates_missing"] = (
    df[["lat", "lon"]]
    .isna()
    .any(axis=1)
)

df[["features_missing", "coordinates_missing"]].sum()

features_missing       1326
coordinates_missing       4
dtype: int64

Missing rent, bond, property counts, dates and coordinates are preserved rather than imputed or replaced with zero. Downstream analyses should filter only the variables they require. Missing feature information is recorded separately from the feature indicators.

## Virtual tour

In [93]:
df["virtual_tour"].value_counts(dropna=False)

virtual_tour
False    11907
True       806
NaN          4
Name: count, dtype: int64

In [94]:
assert df["virtual_tour"].dropna().isin([True, False]).all()
df["virtual_tour"] = df["virtual_tour"].astype("boolean")
df["virtual_tour"].value_counts(dropna=False)

virtual_tour
False    11907
True       806
<NA>         4
Name: count, dtype: Int64

Virtual tour contains only true, false and missing values. Nullable boolean preserves all three states.

## Structured features

In [95]:
df["structured_features"].dropna().head(10)

0     Built in wardrobes, Secure Parking, Bath, Heat...
1               Internal Laundry, Pets Allowed, Heating
2     Internal Laundry, Balcony / Deck, Floorboards,...
3     Air conditioning, Built in wardrobes, Balcony ...
4                           Built in wardrobes, Heating
6     Air conditioning, Heating, Bush Retreat, Ensui...
7                                      Internal Laundry
8     Broadband internet access, Air conditioning, H...
9                 Internal Laundry, Bath, Heating, Shed
10                                        Bath, Heating
Name: structured_features, dtype: object

In [96]:
print("Missing feature records:", df["structured_features"].isna().sum())
print(f"Missing features: {df['structured_features'].isna().mean():.2%}")

raw_feature_labels = (
    df["structured_features"]
    .dropna()
    .str.split(",")
    .explode()
)

raw_feature_labels.value_counts().head(30)

Missing feature records: 1326
Missing features: 10.43%


structured_features
 Dishwasher              5262
 Heating                 5103
 Built in wardrobes      4658
 Secure Parking          3616
 Internal Laundry        2366
 Balcony / Deck          2338
 Air conditioning        2054
Built in wardrobes       2032
 Floorboards             1801
Air conditioning         1742
 Bath                    1642
Internal Laundry         1627
 Intercom                1473
Gas                      1453
 Ensuite                 1425
 Fully fenced            1351
Secure Parking           1350
 Study                   1176
 Gas                      968
 Close to shops           963
 Furnished                951
 Close to schools         929
 Close to transport       901
 Remote Garage            881
 Pets Allowed             872
 Shed                     867
 Ducted Heating           852
 Garden / Courtyard       778
 Split System Heating     760
 Gym                      654
Name: count, dtype: int64

In [97]:
def normalise_feature_name(feature):
    feature = feature.strip().lower()
    feature = feature.replace("&", "and")
    feature = re.sub(r"[/\-]", " ", feature)
    feature = re.sub(r"\s+", " ", feature)
    return feature.strip()

normalised_feature_labels = raw_feature_labels.map(normalise_feature_name)
normalised_feature_labels.value_counts().head(60)

structured_features
built in wardrobes                         6690
heating                                    5710
dishwasher                                 5456
secure parking                             4966
internal laundry                           3993
air conditioning                           3796
balcony deck                               2509
gas                                        2421
floorboards                                1904
bath                                       1881
intercom                                   1629
ensuite                                    1621
fully fenced                               1433
study                                      1280
furnished                                  1128
pets allowed                               1047
close to shops                              965
close to transport                          947
split system heating                        933
close to schools                            932
remote garage       

In [98]:
feature_aliases = {
    "split system air con": "split system air conditioning",
    "split system aircon": "split system air conditioning",
    "split system airconditioning": "split system air conditioning",
    "air conditioner": "air conditioning",
    "broadband": "broadband internet access",
    "broadband internet available": "broadband internet access",
    "broadband connection": "broadband internet access",
    "deck balcony": "balcony deck"
}

features_clean = []

for features in df["structured_features"]:
    if pd.isna(features):
        features_clean.append(None)
    else:
        cleaned = []
        for feature in features.split(","):
            feature = normalise_feature_name(feature)
            feature = feature_aliases.get(feature, feature)
            if feature and feature not in cleaned:
                cleaned.append(feature)
        features_clean.append(sorted(cleaned))

df["features_clean"] = features_clean

feature_counts = df["features_clean"].explode().value_counts()
print("Raw feature labels:", raw_feature_labels.nunique())
print("Normalised feature labels:", normalised_feature_labels.nunique())
print("Clean feature labels:", len(feature_counts))
feature_counts.head(60)

Raw feature labels: 653
Normalised feature labels: 521
Clean feature labels: 513


features_clean
built in wardrobes                          6690
heating                                     5710
dishwasher                                  5456
secure parking                              4966
internal laundry                            3993
air conditioning                            3920
balcony deck                                2510
gas                                         2421
floorboards                                 1904
bath                                        1881
intercom                                    1629
ensuite                                     1621
fully fenced                                1433
study                                       1280
furnished                                   1128
pets allowed                                1047
split system air conditioning               1023
close to shops                               965
close to transport                           947
close to schools                             932
split

Case, spacing and slash/hyphen variants are normalised for labels such as built-in wardrobes, balcony/deck, secure parking and internal laundry. The alias map merges only clear wording variants. Balcony, deck and the combined balcony/deck label remain separate, as do general air conditioning and specific systems. Each cleaned feature is counted once per listing.

In [99]:
MIN_FEATURE_COUNT = 100
common_features = feature_counts[feature_counts >= MIN_FEATURE_COUNT].index
feature_columns = []

for feature in common_features:
    column = "feature_" + re.sub(r"[^a-z0-9]+", "_", feature).strip("_")
    assert column not in df.columns, f"Duplicate feature column: {column}"
    df[column] = df["features_clean"].apply(
        lambda features: int(features is not None and feature in features)
    )
    feature_columns.append(column)

print("Feature indicator columns:", len(feature_columns))
df[feature_columns].sum().sort_values(ascending=False)

Feature indicator columns: 49


feature_built_in_wardrobes               6690
feature_heating                          5710
feature_dishwasher                       5456
feature_secure_parking                   4966
feature_internal_laundry                 3993
feature_air_conditioning                 3920
feature_balcony_deck                     2510
feature_gas                              2421
feature_floorboards                      1904
feature_bath                             1881
feature_intercom                         1629
feature_ensuite                          1621
feature_fully_fenced                     1433
feature_study                            1280
feature_furnished                        1128
feature_pets_allowed                     1047
feature_split_system_air_conditioning    1023
feature_close_to_shops                    965
feature_close_to_transport                947
feature_close_to_schools                  932
feature_split_system_heating              932
feature_broadband_internet_access 

There were 653 raw feature labels. Case, punctuation and spacing normalisation reduced this to 521, and explicit aliases reduced it to 513. Only 49 features appear in at least 100 listings, avoiding hundreds of sparse indicator columns.

A zero indicator means the feature was not recorded, rather than confirmed absent. For the listings with missing feature information, all indicators are zero and `features_missing` is true. The original `structured_features` and cleaned lists are retained for traceability, including rare labels.

## Final missingness and schema check

In [100]:
# Lists are converted to tuples only for counting unique combinations.
unique_counts = df.drop(columns=["features_clean"]).nunique(dropna=True)
unique_counts["features_clean"] = df["features_clean"].apply(
    lambda features: tuple(features) if features is not None else None
).nunique(dropna=True)

missingness_summary = pd.DataFrame({
    "dtype": df.dtypes,
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique": unique_counts
}).sort_values("missing_pct", ascending=False)

missingness_summary

,dtype,missing,missing_pct,unique
property_id,object,10145,79.78,2555
carspaces,Int64,1803,14.18,17
features_clean,object,1326,10.43,6866
structured_features,object,1326,10.43,7320
flag_property_features_suspicious,boolean,1248,9.81,2
...,...,...,...,...
postcode,string[python],0,0.00,426
property_type,string[python],0,0.00,16
scraped_date,datetime64[ns],0,0.00,935
suburb,string[python],0,0.00,656


In [101]:
missingness_summary.loc[
    missingness_summary["missing"] > 0
].head(20)

,dtype,missing,missing_pct,unique
property_id,object,10145,79.78,2555
carspaces,Int64,1803,14.18,17
features_clean,object,1326,10.43,6866
structured_features,object,1326,10.43,7320
flag_property_features_suspicious,boolean,1248,9.81,2
flag_carspaces_suspicious,boolean,1182,9.29,2
bond,float64,782,6.15,983
weekly_rent,float64,279,2.19,408
agent_names,string[python],248,1.95,3586
available_date,datetime64[ns],136,1.07,734


Property identifiers, feature records and some property counts remain incomplete. Smaller gaps in rent, bond, dates and coordinates are also preserved. Neither imputation nor dropping these listings is justified at this stage; `property_id` and `domain_page_id` remain available for later checks of repeated physical properties.

## Provenance

In [102]:
df["source"] = "Domain.com.au"
df["dataset_snapshot"] = pd.Timestamp("2025-09-09")
# Use nanosecond precision so the dtype is preserved in Parquet.
df["dataset_snapshot"] = df["dataset_snapshot"].astype("datetime64[ns]")

## Integrity checks

In [103]:
assert df["listing_id"].notna().all()
assert df["listing_id"].is_unique
assert len(df) == len(raw)

# Use tuples in a temporary copy because list values cannot be hashed.
duplicate_check = df.copy()
duplicate_check["features_clean"] = duplicate_check["features_clean"].apply(
    lambda features: tuple(features) if features is not None else None
)
assert not duplicate_check.duplicated().any()
del duplicate_check

assert df[feature_columns].isin([0, 1]).all().all()
print(f"Integrity checks passed: {len(df):,} unique listings retained.")

Integrity checks passed: 12,717 unique listings retained.


## Deterministic ordering

In [106]:
df = (
    df
    .sort_values("listing_id")
    .reset_index(drop=True)
)

df.head(30)

,listing_id,suburb,postcode,weekly_rent,bond,available_date,date_listed,days_listed,bedrooms,bathrooms,...,feature_rumpus_room,feature_ducted_cooling,feature_city_views,feature_gas_heating,feature_deck,feature_laundry,feature_separate_dining_room,feature_double_glazed_windows,source,dataset_snapshot
0,5470976,ASCOT VALE,3032,660.0,2868.0,2025-09-10,2025-07-21,50,2,1,...,0,0,0,0,0,0,0,0,Domain.com.au,2025-09-09
1,5604062,MELTON,3337,NaN,150.0,2008-01-15,2008-01-15,6446,<NA>,<NA>,...,0,0,0,0,0,0,0,0,Domain.com.au,2025-09-09
2,6168570,MELBOURNE,3000,310.0,1347.0,2023-07-03,2025-09-02,7,<NA>,1,...,0,0,1,0,0,0,0,1,Domain.com.au,2025-09-09
3,7117948,MOONEE PONDS,3039,500.0,2173.0,2025-09-19,2025-09-06,3,2,1,...,0,0,0,0,0,0,0,0,Domain.com.au,2025-09-09
4,7455074,PRAHRAN,3181,NaN,340.0,2014-05-22,2012-03-02,4938,1,1,...,0,0,0,0,0,0,0,0,Domain.com.au,2025-09-09
5,7625736,BRIGHTON,3186,1575.0,NaN,2022-06-18,2012-06-09,4839,2,2,...,0,1,0,0,0,1,0,0,Domain.com.au,2025-09-09
6,8074820,QUARRY HILL,3550,650.0,2824.0,2026-07-21,2012-12-12,4654,2,2,...,0,0,0,0,0,0,0,0,Domain.com.au,2025-09-09
7,8253224,BENDIGO,3550,580.0,2520.0,2026-01-20,2013-03-25,4551,2,1,...,0,0,0,0,0,0,0,0,Domain.com.au,2025-09-09
8,8303216,ARMADALE,3143,1575.0,NaN,2022-06-18,2013-04-22,4522,2,2,...,0,1,0,0,0,1,0,0,Domain.com.au,2025-09-09
9,8430042,CLAYTON,3168,360.0,NaN,2018-07-20,2023-03-02,922,1,1,...,0,0,0,0,0,0,0,0,Domain.com.au,2025-09-09


## Save curated data

In [105]:
df.to_parquet(
    CURATED_PATH,
    index=False
)

curated = pd.read_parquet(CURATED_PATH)

assert curated.shape[0] == df.shape[0]
assert curated.shape[1] == df.shape[1]
assert curated.columns.equals(df.columns)
assert curated["listing_id"].equals(df["listing_id"])
assert curated["listing_id"].is_monotonic_increasing
assert curated["listing_id"].notna().all()
assert curated["listing_id"].is_unique

key_columns = [
    "listing_id", "property_id", "domain_page_id", "postcode",
    "weekly_rent", "bond", "bedrooms", "bathrooms", "carspaces",
    "date_listed", "available_date", "scraped_date", "virtual_tour",
    "lat", "lon", "features_missing", "coordinates_missing",
    "flag_bad_coordinates", "dataset_snapshot"
] + property_flags + feature_columns

assert curated[key_columns].dtypes.equals(df[key_columns].dtypes)
assert curated.isna().equals(df.isna())
# Parquet reloads list values as arrays; compare their contents as lists.
assert curated["features_clean"].apply(
    lambda features: list(features) if features is not None else None
).equals(df["features_clean"])

print(f"Saved and verified {curated.shape[0]:,} listings and "
      f"{curated.shape[1]} columns: {CURATED_PATH}")

Saved and verified 12,717 listings and 94 columns: ../data/curated/vic_rentals.parquet


## Curation summary

The curated dataset retains one row per Domain rental listing, identified by `listing_id`. No rows were arbitrarily deleted for unusual values; explicit quality flags preserve suspicious observations for later review. Land area was dropped because almost all values were missing.

Structured property features were standardised and selectively one-hot encoded, with the original text retained. Missing values were preserved rather than imputed. Coordinates were validated, with SA2 mapping deferred to the dedicated geospatial notebook.

The final output is `data/curated/vic_rentals.parquet`. This is restricted course data and must not be committed to Git or redistributed. Notebook outputs containing listing-level data must also remain private.